# Migrating from version 1

Version 2 is a rewrite. The calculation is the same idea, but the interface, the error estimate and the three-point structure function have changed. This page maps each part of the version 1 call onto version 2; the version 2 code runs on the Mao et al. (2010) data bundled with the package.

## The call

Version 1:

```python
sf_result, fit_result = structure_function(
    data=rm,                # Quantity
    errors=rm_err,          # Quantity
    coords=coords,          # SkyCoord in ICRS
    samples=1000,           # Monte-Carlo samples for the errors
    bins=bins,
    show_plots=True,
    save_plots=True,
    verbose=True,
    fit="bilby",            # or "lsq", "lsq_weight", None
    outdir="outdir",
    model_name="broken_power_law",
    n_point=2,
    sampler="nestle",       # passed on to bilby
    nlive=400,
)
```

Version 2 splits this into computing, fitting, plotting and saving:

In [ ]:
from importlib import resources

import astropy.units as u
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table

from structurefunction import fit_sf, structure_function, write
from structurefunction.plotting import plot_sf

with resources.as_file(
    resources.files("structurefunction.data") / "mao2010_ngp.ecsv"
) as path:
    table = Table.read(path)
table = table[table["used_s5_2"]]
rm = table["rm"].quantity
rm_err = table["rm_err"].quantity
coords = SkyCoord(table["ra"], table["dec"])
bins = 10 ** np.arange(-0.8, 1.6, 0.1) * u.deg

sf_result = structure_function(rm, rm_err, coords, bins=bins, jk_regions=70, rng=1)
fit_result = fit_sf(sf_result, model="broken_power_law", sampler="nautilus", rng=2)
plot_sf(sf_result, fit=fit_result)
write(sf_result, "sf.ecsv", overwrite=True)

| Version 1 | Version 2 |
|---|---|
| `data` | `values`: plain array or `Quantity` |
| `errors` | `errors`: plain array, `Quantity` or `None` |
| `coords` | `coords`: any `SkyCoord` frame; with distances, 3D separations |
| `samples` | removed: errors come from jackknife regions, `jk_regions` (see below) |
| `bins` | `bins`: number of log bins or edges; plus `min_sep`, `max_sep` |
| `show_plots`, `save_plots`, `outdir` | `plot_sf(result)` and `write(result, path)` |
| `verbose` | the `structurefunction` logger (`logging.getLogger("structurefunction")`) |
| `fit`, `model_name`, `**kwargs` | `fit_sf(result, model=..., sampler=...)` or `compare_models(result)` |
| `n_point=3` | `n_point=3` with `stencil=StencilOptions(...)` (definition changed, see below) |
| returns `(SFResult, bilby Result)` | returns an `SFResult`; fits are separate |

## The result

| Version 1 `SFResult` | Version 2 `SFResult` |
|---|---|
| `med` | `sf` (plain array; unit in `sf_unit`) |
| `err_low`, `err_high` | `sf_err` and the full covariance `cov` |
| `count` | `count` |
| `c_bins` (bin midpoints) | `separation` (mean separation of the pairs in each bin); `bin_edges` |
| | `noise_bias`, `saturation`, `reliable`, `resampling_reliable`, `region_size` |

In [ ]:
print(sf_result.sf_unit)
to_show = Table(
    {
        "separation": sf_result.separation,
        "sf": sf_result.sf * sf_result.sf_unit,
        "sf_err": sf_result.sf_err * sf_result.sf_unit,
        "count": sf_result.count,
    }
)
to_show[sf_result.reliable]

## What changes in the numbers

**Error correction.** Version 1 drew Monte-Carlo samples of the data and of the errors and subtracted the mean squared difference of the sampled errors. That removed only part of the measurement-error contribution. Version 2 subtracts $\sigma_1^2 + \sigma_2^2$ for every pair, which is exact for independent errors. Structure functions are therefore lower than in version 1 by about the mean error variance of a pair; `noise_bias` holds what was subtracted.

In [ ]:
mean_removed = np.nanmean(sf_result.noise_bias)
print(f"Mean error term removed: {mean_removed:.0f} {sf_result.sf_unit}")

**Error bars.** Version 1 error bars came only from the measurement errors. Version 2 uses jackknife regions: the sources are split into compact areas of sky of similar size, and the structure function is recomputed with each left out in turn. The error bars then include the variation between parts of the sky, and `cov` gives the correlation between bins, which the fits use. Choose `jk_regions` to be several times the number of bins, with each region larger than the separations of interest.

**Three-point structure function.** Version 1 combined any two pairs that shared a source, whatever their directions. Version 2 uses the stencil of Seta et al. (2022): the outer sources must lie on opposite sides of the middle one, within `angle_tol_deg` of a straight line, at distances that agree to within `sep_tol`. The values differ from version 1 and follow the published definition.

**Bins.** `c_bins` were the arithmetic midpoints of the bins; `separation` is the mean separation of the pairs actually in each bin.

## Fitting

Version 1 fitted with bilby or least squares and treated bins as independent. Version 2 fits with the full covariance between bins and offers MAP, emcee and nautilus (with evidence). The models are `constant`, `power_law`, `broken_power_law` and `smooth_broken_power_law`, and `compare_models` ranks them. bilby is no longer a dependency.

## Installation

Version 2 needs Python 3.12 or later. Plotting needs the `plot` extra and nautilus the `evidence` extra:

```bash
pip install "structurefunction[plot,evidence]"
```